# Fundamentals of Machine Learning — Exercise 7

## Gradient Boosting: Learning by Correcting Previous Errors

In Exercise 6, a Random Forest trained many trees largely **in parallel** and combined their predictions. Gradient boosting follows a different strategy:

> **Build a sequence of small models, where each new model focuses on what the current ensemble still gets wrong.**

This exercise moves from one transparent residual update to practical classification with validation curves and early stopping.

The classroom workflow remains:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

After completing this exercise, you should be able to:

- explain boosting as an additive sequence of weak learners,
- calculate and interpret residuals in a simple regression example,
- explain the role of `learning_rate` and `n_estimators`,
- distinguish training improvement from validation improvement,
- recognise underfitting and overfitting from staged curves,
- compare a single tree, Random Forest, Gradient Boosting, and Histogram Gradient Boosting,
- use validation data to select the number of boosting stages,
- keep the test set untouched until the final model decision,
- justify a boosting configuration under a simple resource constraint,
- use an AI chatbot to question an already written diagnosis rather than generate it.

**Recommended duration:** 100–120 minutes.  
The final model-selection memo may be completed as homework.

## How to work in this exercise

Most routine code is prepared. Your main tasks are to:

1. predict what a new boosting stage should change,
2. calculate one small example manually,
3. change only clearly marked settings,
4. interpret training and validation curves,
5. justify a model decision using evidence,
6. verify that the test set was not used during model selection.

A high training score is not the goal. The goal is a model that generalises to unseen data.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import (
    load_breast_cancer,
    make_friedman1,
    make_moons,
)
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42


## 1. Reading: Gradient Boosting - learning by correcting errors

### The central idea

In the previous exercise, we worked with **Decision Trees** and **Random Forests**.

Random Forest combines predictions from multiple trees trained largely independently. **Gradient Boosting takes a different approach: it builds trees sequentially, with each new tree attempting to correct the errors of the current ensemble.**

Instead of training one complex model, we gradually construct a prediction function by adding many relatively simple models.

Let us demonstrate this principle using a regression problem.

### Step 1: Start with a simple prediction

Suppose we want to predict house prices based on their floor area.

Our training dataset contains four houses:

| Area ($m^2$) | Actual price (million) |
|---|---|
| 50 | 3 |
| 70 | 4 |
| 90 | 6 |
| 110 | 8 |

For regression with squared error, a convenient initial model is the mean of the target values:

$$
F_0(x) = \bar{y}.
$$

In our example:

$$
F_0(x) = \frac{3+4+6+8}{4}=5.25
$$

The initial model predicts **5.25 million for every house**, regardless of its area.

Obviously, this is not a very useful model yet.

### Step 2: Calculate the prediction errors

We calculate the difference between each actual value and its prediction. This difference is called a **residual**:

$$
r_i = y_i - F_0(x_i).
$$

| Area | Actual price | Prediction | Residual |
|---|---|---|---|
| 50 | 3 | 5.25 | -2.25 |
| 70 | 4 | 5.25 | -1.25 |
| 90 | 6 | 5.25 | +0.75 |
| 110 | 8 | 5.25 | +2.75 |

Notice that:

- A **positive residual** means that the model underestimates the actual value.
- A **negative residual** means that the model overestimates the actual value.

Our objective is now to correct these errors.

### Step 3: Train a decision tree to predict the residuals

We train a small regression tree, but with one important difference:

**The target variable is no longer the house price. It is the residual of the current model.**

The training data for the first tree therefore becomes:

| Input: Area | Target: Residual |
|---|---|
| 50 | -2.25 |
| 70 | -1.25 |
| 90 | +0.75 |
| 110 | +2.75 |

Suppose we train a decision tree with a maximum depth of 1.

Such a tree can perform only one split. For example:

```text
                 Area <= 80?
                  /       \
                Yes        No
                 |          |
               -1.75      +1.75
```

The values in the leaves are the mean residuals of the corresponding observations.

The tree therefore predicts:

$$
h_1(x)=
\begin{cases}
-1.75 & x \leq 80,\\
+1.75 & x > 80.
\end{cases}
$$

Notice that this tree **does not predict house prices**. It predicts how the current model's predictions should be corrected.

### Step 4: Update the model

We combine the initial prediction with the correction predicted by the tree:

$$
F_1(x)=F_0(x)+\eta h_1(x),
$$

where:

- $F_0(x)$ is the initial model,
- $h_1(x)$ is the first regression tree,
- $\eta$ is the **learning rate**, controlling the size of the correction.

For simplicity, let us initially use $\eta=1$.

For a house with an area of 50 $m^2$:

$$
F_1(50)=5.25+(-1.75)=3.5
$$

For a house with an area of 110 $m^2$:

$$
F_1(110)=5.25+1.75=7.0
$$

The updated predictions are:

| Area | Actual price | Initial prediction | Updated prediction |
|---|---|---|---|
| 50 | 3 | 5.25 | 3.5 |
| 70 | 4 | 5.25 | 3.5 |
| 90 | 6 | 5.25 | 7.0 |
| 110 | 8 | 5.25 | 7.0 |

Our model has improved, but its predictions are still not perfect.

### Step 5: Train another tree

We calculate the residuals again, this time using the updated predictions:

$$
r_i^{(2)}=y_i-F_1(x_i).
$$

| Area | Actual price | Current prediction | New residual |
|---|---|---|---|
| 50 | 3 | 3.5 | -0.5 |
| 70 | 4 | 3.5 | +0.5 |
| 90 | 6 | 7.0 | -1.0 |
| 110 | 8 | 7.0 | +1.0 |

We now train **a new decision tree** using the original input features and these new residuals.

The tree searches for a split that minimizes the squared error of its residual predictions.

For example, one optimal split for this dataset is:

```text
                 Area <= 60?
                  /       \
                Yes        No
                 |          |
               -0.5       +0.167
```

The resulting model is:

$$
F_2(x)=F_1(x)+\eta h_2(x).
$$

For $\eta=1$, our updated predictions become approximately:

| Area | Actual price | Prediction after first tree | Prediction after second tree |
|---|---|---|---|
| 50 | 3 | 3.5 | 3.0 |
| 70 | 4 | 3.5 | 3.667 |
| 90 | 6 | 7.0 | 7.167 |
| 110 | 8 | 7.0 | 7.167 |

The model has improved again.

**Importantly, the second tree does not modify the first tree. It is a separate model trained to correct the errors of the entire current ensemble.**

### Step 6: Repeat the process

We can continue adding trees in the same way.

After $M$ iterations, the resulting model is:

$$
F_M(x)=F_0(x)+\eta\sum_{m=1}^{M}h_m(x).
$$

Each tree contributes a correction to the previous ensemble.

This introduces two important hyperparameters:

- **`n_estimators`** — the number of trees added to the ensemble.
- **`learning_rate`** — the contribution of each new tree to the current prediction.

A smaller learning rate means smaller corrections, so more trees are often needed.

However, adding more trees does not necessarily improve performance on unseen data. Eventually, the ensemble may begin to overfit the training dataset.

### Why is it called Gradient Boosting?

So far, we have trained trees to predict residuals. But where does the word *gradient* come from?

Consider the squared error loss:

$$
L(y,F(x))=\frac{1}{2}(y-F(x))^2.
$$

Its derivative with respect to the prediction is:

$$
\frac{\partial L}{\partial F(x)}=F(x)-y.
$$

Therefore, the negative gradient is:

$$
-\frac{\partial L}{\partial F(x)}=y-F(x).
$$

This is exactly the residual we have been using.

**By training a tree to predict residuals, we are approximating the negative gradient of the loss function.**

For other loss functions, such as logistic loss in classification, the negative gradients are not necessarily ordinary residuals. They are often called *pseudo-residuals*.

The general idea remains the same: each new tree helps move the current model in a direction that reduces the loss.

### Key takeaways

1. Gradient Boosting builds an ensemble of trees **sequentially**.
2. Each new tree is trained to correct the errors of the **current ensemble**.
3. The original input features remain the same, but the training targets for new trees change.
4. `n_estimators` controls the number of boosting iterations.
5. `learning_rate` controls the size of each correction.
6. For squared error regression, the correction targets are the residuals, which correspond to the negative gradient of the loss.

In [ ]:
x_toy = np.array([0, 1, 2, 3, 4, 5], dtype=float)
y_toy = np.array([1.0, 1.4, 1.8, 4.2, 4.6, 5.0])

initial_prediction = np.repeat(y_toy.mean(), len(y_toy))
initial_residual = y_toy - initial_prediction

toy_table = pd.DataFrame({
    'x': x_toy,
    'observed y': y_toy,
    'initial prediction': initial_prediction,
    'residual y - prediction': initial_residual,
})

display(toy_table.round(3))
print('Initial MSE:', round(mean_squared_error(y_toy, initial_prediction), 4))

### Activity 1 - Read the residuals

Without fitting a tree yet:

1. Which observations need their prediction increased?
2. Which observations need their prediction decreased?
3. Around which value of `x` would a one-split tree probably separate the residuals?
4. Why is the residual not the final target variable of the original problem?

### Fit one decision stump to the residuals

A stump is a decision tree with `max_depth=1`. It can create only one split.

In [ ]:
LEARNING_RATE = 0.50

first_stump = DecisionTreeRegressor(
    max_depth=1,
    random_state=RANDOM_STATE,
)
first_stump.fit(x_toy.reshape(-1, 1), initial_residual)

stump_correction = first_stump.predict(x_toy.reshape(-1, 1))
updated_prediction = initial_prediction + LEARNING_RATE * stump_correction
updated_residual = y_toy - updated_prediction

first_stage_table = pd.DataFrame({
    'x': x_toy,
    'observed y': y_toy,
    'before': initial_prediction,
    'residual before': initial_residual,
    'stump correction': stump_correction,
    'after update': updated_prediction,
    'residual after': updated_residual,
})

display(first_stage_table.round(3))
print('MSE before:', round(mean_squared_error(y_toy, initial_prediction), 4))
print('MSE after: ', round(mean_squared_error(y_toy, updated_prediction), 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(x_toy, y_toy, s=75, label='Observed values')
plt.plot(x_toy, initial_prediction, marker='o', label='Initial constant model')
plt.plot(x_toy, updated_prediction, marker='o', label='After one stump')
plt.xlabel('x')
plt.ylabel('y')
plt.title('One boosting update')
plt.legend()
plt.show()

### Stop and explain

Complete the argument:

> The first stump does not predict `y` directly. It predicts __________.  
> Multiplying by the learning rate causes __________.  
> After the update, the residuals change because __________.

What would happen if `LEARNING_RATE = 0`?  
What risk might arise if the learning rate were extremely large?

## 2. Repeating the process creates an additive model

The next function repeats the same three operations:

1. compute residuals,
2. fit a shallow tree,
3. add a shrunken correction.

Read the loop and identify where each operation occurs.

In [ ]:
def fit_simple_gradient_boosting(x, y, n_stages=10, learning_rate=0.20):
    x = np.asarray(x).reshape(-1, 1)
    y = np.asarray(y)

    prediction = np.repeat(y.mean(), len(y)).astype(float)
    history = [prediction.copy()]
    trees = []

    for _ in range(n_stages):
        residual = y - prediction

        tree = DecisionTreeRegressor(
            max_depth=1,
            random_state=RANDOM_STATE,
        )
        tree.fit(x, residual)

        correction = tree.predict(x)
        prediction = prediction + learning_rate * correction

        trees.append(tree)
        history.append(prediction.copy())

    return trees, history

_, toy_history = fit_simple_gradient_boosting(
    x_toy,
    y_toy,
    n_stages=20,
    learning_rate=0.20,
)

stage_mse = [
    mean_squared_error(y_toy, prediction)
    for prediction in toy_history
]

pd.DataFrame({
    'stage': range(len(stage_mse)),
    'training MSE': stage_mse,
}).head(10).round(4)

In [ ]:
selected_stages = [0, 1, 3, 10, 20]

plt.figure(figsize=(9, 5))
plt.scatter(x_toy, y_toy, s=80, label='Observed values')

for stage in selected_stages:
    plt.plot(
        x_toy,
        toy_history[stage],
        marker='o',
        label=f'Stage {stage}',
    )

plt.xlabel('x')
plt.ylabel('y')
plt.title('The additive prediction after selected stages')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(range(len(stage_mse)), stage_mse, marker='o')
plt.xlabel('Boosting stage')
plt.ylabel('Training MSE')
plt.title('Training error on the toy data')
plt.show()

### Activity 2 - What becomes more complex?

1. Each individual stump is simple. Why can their sum represent a more complex function?
2. Why does training error usually decrease as stages are added?
3. Why does decreasing training error not guarantee better performance on new data?

## 3. Learning rate and number of trees must be considered together

A smaller learning rate makes each update more conservative, but usually requires more trees. A larger learning rate learns faster, but may reach overfitting earlier.

We now compare several configurations on a nonlinear regression dataset.

In [ ]:
X_reg, y_reg = make_friedman1(
    n_samples=800,
    n_features=6,
    noise=1.2,
    random_state=RANDOM_STATE,
)

X_reg_train, X_reg_valid, y_reg_train, y_reg_valid = train_test_split(
    X_reg,
    y_reg,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

regression_configs = {
    'slow: lr=0.03, 350 trees': {
        'learning_rate': 0.03,
        'n_estimators': 350,
    },
    'balanced: lr=0.10, 180 trees': {
        'learning_rate': 0.10,
        'n_estimators': 180,
    },
    'aggressive: lr=0.50, 80 trees': {
        'learning_rate': 0.50,
        'n_estimators': 80,
    },
}

regression_models = {}
regression_curve_rows = []

plt.figure(figsize=(10, 6))

for label, params in regression_configs.items():
    model = GradientBoostingRegressor(
        max_depth=2,
        random_state=RANDOM_STATE,
        **params,
    )
    model.fit(X_reg_train, y_reg_train)
    regression_models[label] = model

    valid_mse_curve = [
        mean_squared_error(y_reg_valid, prediction)
        for prediction in model.staged_predict(X_reg_valid)
    ]

    best_stage = int(np.argmin(valid_mse_curve) + 1)
    best_valid_mse = float(np.min(valid_mse_curve))

    regression_curve_rows.append({
        'configuration': label,
        'best validation stage': best_stage,
        'best validation MSE': best_valid_mse,
        'final validation MSE': valid_mse_curve[-1],
    })

    plt.plot(
        range(1, len(valid_mse_curve) + 1),
        valid_mse_curve,
        label=label,
    )

plt.xlabel('Number of boosting stages')
plt.ylabel('Validation MSE')
plt.title('Validation error across boosting stages')
plt.legend()
plt.show()

pd.DataFrame(regression_curve_rows).round(3)

### Activity 3 - Diagnose the curves

For each configuration, identify:

- whether learning is slow or fast,
- approximately where the best validation result occurs,
- whether later stages continue to help or begin to hurt.

Then answer:

> Why is it incomplete to state that “a learning rate of 0.03 is better than 0.10” without mentioning the number of trees?


## 4. Reading: Gradient Boosting for classification

### From regression to classification

So far, we have used Gradient Boosting for regression, where the goal was to predict a continuous value.

For classification, the objective is different. We want to predict which class an observation belongs to.

For binary classification, the target variable is:

$$
y \in \{0,1\}.
$$

The main principle of Gradient Boosting remains the same: **each new tree attempts to improve the predictions of the current ensemble.**

However, instead of minimizing squared error, we typically minimize a classification loss, such as **log loss**.

### From model scores to probabilities

In binary classification, Gradient Boosting constructs an additive model:

$$
F_M(x)=F_0(x)+\eta\sum_{m=1}^{M}h_m(x).
$$

Unlike regression, the output $F_M(x)$ is not directly the predicted class or probability. It is a **raw prediction score** (log-odds).

We convert this score into a probability using the sigmoid function:

$$
p(x)=\frac{1}{1+e^{-F_M(x)}}.
$$

The predicted class can then be determined using a threshold, typically 0.5:

$$
\hat{y}=
\begin{cases}
1 & \text{if }p(x)\geq 0.5,\\
0 & \text{otherwise}.
\end{cases}
$$

### How does the model learn?

For binary classification, the log loss is:

$$
L(y,p)=-y\log(p)-(1-y)\log(1-p).
$$

The negative gradient with respect to the raw prediction score is:

$$
r_i=y_i-p_i.
$$

These values are called **pseudo-residuals**.

A new decision tree is trained to approximate these pseudo-residuals, helping the ensemble increase probabilities for positive observations and decrease them for negative observations.

The tree contributes corrections to the raw scores, not directly to the probabilities.

### Building a nonlinear decision boundary

Each shallow decision tree divides the feature space into simple regions.

A single tree may produce a relatively simple decision boundary. By adding more trees, Gradient Boosting can combine these regions into increasingly complex boundaries.

For example, when classifying the two interleaving moons in the `make_moons` dataset:

- With **1 boosting stage**, the model can capture only a simple approximation of the class boundary.
- With **10 stages**, the boundary can better follow the nonlinear structure of the data.
- With **100 stages**, the model may capture more details, but it can also begin fitting noise.

More trees therefore do not automatically guarantee better generalization.

### Predict before running

How do you expect the decision boundary to change as the number of boosting stages increases from 1 to 100?

Will the training accuracy always improve? What about the accuracy on unseen data?


In [ ]:
X_moons, y_moons = make_moons(
    n_samples=600,
    noise=0.25,
    random_state=RANDOM_STATE,
)

X_moon_train, X_moon_test, y_moon_train, y_moon_test = train_test_split(
    X_moons,
    y_moons,
    test_size=0.30,
    stratify=y_moons,
    random_state=RANDOM_STATE,
)

moon_stages = [1, 5, 30, 100]
moon_models = {}
moon_rows = []

for n_stages in moon_stages:
    model = GradientBoostingClassifier(
        n_estimators=n_stages,
        learning_rate=0.10,
        max_depth=1,
        random_state=RANDOM_STATE,
    )
    model.fit(X_moon_train, y_moon_train)
    moon_models[n_stages] = model

    probability = model.predict_proba(X_moon_test)[:, 1]
    prediction = (probability >= 0.5).astype(int)

    moon_rows.append({
        'stages': n_stages,
        'test accuracy': accuracy_score(y_moon_test, prediction),
        'test ROC-AUC': roc_auc_score(y_moon_test, probability),
        'test log loss': log_loss(y_moon_test, probability),
    })

pd.DataFrame(moon_rows).round(3)

In [ ]:
def plot_decision_surface(axis, model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    probability = model.predict_proba(grid)[:, 1].reshape(xx.shape)

    axis.contourf(xx, yy, probability, levels=20, alpha=0.35, cmap='coolwarm')
    axis.contour(xx, yy, probability, levels=[0.5], colors='black', linewidths=1.2)
    axis.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=22, alpha=0.75)
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

for axis, n_stages in zip(axes, moon_stages):
    plot_decision_surface(
        axis,
        moon_models[n_stages],
        X_moon_test,
        y_moon_test,
        f'{n_stages} stage(s)',
    )

plt.tight_layout()
plt.show()

### Stop and explain

1. How can many axis-aligned stump decisions create a curved-looking boundary?
2. Which change in the table suggests that probability quality improved, even when accuracy changed only slightly?
3. Why should we not automatically select the model with the most stages?

## 5. Real-data comparison: breast cancer diagnosis

We reuse the Breast Cancer dataset so that the new method can be compared directly with models from Exercise 6.

The split has three roles:

- **training set:** fit model parameters,
- **validation set:** choose model and number of stages,
- **test set:** final audit after the choice is fixed.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data.copy()
y = cancer.target.copy()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

print('Training rows:  ', len(X_train))
print('Validation rows:', len(X_valid))
print('Test rows:      ', len(X_test))
print('Positive-class rate:', round(float(y.mean()), 3))


## Reading: Histogram Gradient Boosting

### Why do we need a faster Gradient Boosting algorithm?

Traditional Gradient Boosting builds decision trees sequentially. When constructing each tree, the algorithm searches for the best split of the training data.

For numerical features with many distinct values, evaluating possible split thresholds can be computationally expensive, especially for large datasets.

**Histogram Gradient Boosting (HGB)** improves efficiency by grouping continuous feature values into a limited number of discrete intervals, called *bins*.

### How does histogram binning work?

Suppose we have a numerical feature representing house areas:

`[51, 53, 58, 62, 75, 81, 94, 110, 125]`

Instead of evaluating many individual feature values as possible split thresholds, HGB first groups them into bins.

For example:

| Bin | Area interval | Number of houses |
|---|---|---|
| 1 | 50–69 | 4 |
| 2 | 70–89 | 2 |
| 3 | 90–109 | 1 |
| 4 | 110–129 | 2 |

The algorithm can then evaluate splits between these bins rather than between all distinct feature values.

In practice, bin boundaries are typically chosen based on the distribution of feature values, rather than using equally sized intervals.

This substantially reduces the number of candidate splits.

### What stays the same?

Histogram Gradient Boosting still follows the same fundamental principle:

1. Start with an initial prediction.
2. Calculate gradients of the loss function.
3. Train a new tree to improve the current model.
4. Add the tree's contribution to the ensemble.
5. Repeat the process.

**The main difference is how decision trees are constructed, not the fundamental boosting principle.**

### Important hyperparameters

In scikit-learn, `HistGradientBoostingClassifier` provides several useful parameters:

- `max_iter` - maximum number of boosting iterations (analogous to `n_estimators`).
- `learning_rate` - controls the contribution of each boosting iteration.
- `max_leaf_nodes` - limits the complexity of individual trees.
- `max_bins` - maximum number of bins used for numerical features.
- `early_stopping` - allows training to stop when validation performance no longer improves sufficiently.

### Early stopping

Adding more trees can reduce training error, but it may eventually lead to overfitting.

With **early stopping**, the algorithm monitors performance on validation data and stops adding trees when sufficient improvement is no longer observed.

This can reduce training time and help prevent overfitting.

### Key takeaway

Histogram Gradient Boosting is an efficient implementation of Gradient Boosting designed particularly for larger datasets.

It achieves its computational advantage by **evaluating tree splits using binned feature values instead of all individual values**, while preserving the sequential error-correction principle of Gradient Boosting.

### Check your understanding

1. Why can histogram binning make tree construction faster?
2. Why might early stopping be useful when training a model with a large `max_iter`?



## Reading: Training Loss vs. Validation Loss

### Why do we need validation loss?

In Gradient Boosting, each new tree attempts to improve the predictions of the current ensemble.

As we add more trees, the model becomes increasingly flexible and can fit the training data more closely.

However, **a model that performs well on training data does not necessarily perform well on unseen data**.

To evaluate how well the model generalizes, we monitor two quantities:

- **Training loss** — measures prediction errors on the data used to train the model.
- **Validation loss** — measures prediction errors on a separate dataset that was not used to fit the model.

Both should be calculated using the same loss function, such as log loss for classification.

### How do the losses change during training?

As the number of boosting iterations increases, we typically observe three stages:

1. **Underfitting:** With only a few trees, the model is too simple. Both training and validation losses are relatively high.
2. **Good generalization:** Additional trees improve predictions on both datasets. Validation loss decreases.
3. **Overfitting:** The model begins fitting noise or specific patterns in the training data. Training loss continues decreasing, while validation loss starts increasing.

The typical behavior is:

```text
Loss
 ^
 |\
 | \       Validation loss
 |  \       ________
 |   \_____/        \____
 |                      \
 | \                     Training loss
 |  \________________________
 |
 +------------------------------> Number of trees
             ^
       Best validation
          performance
```

*Conceptual illustration: the validation loss may reach a minimum and then increase, while training loss continues to decrease.*

### Choosing the number of trees

The number of boosting iterations is controlled by:

- `n_estimators` in `GradientBoostingClassifier`
- `max_iter` in `HistGradientBoostingClassifier`

Instead of automatically using the maximum number of trees, we can select the number of iterations that achieves the **lowest validation loss**:

$$
M^* = \arg\min_M L_{\mathrm{val}}(M)
$$

where:

- $M$ is the number of boosting iterations,
- $L_{\mathrm{val}}(M)$ is the validation loss after $M$ iterations,
- $M^*$ is the selected number of iterations.

This approach helps balance model complexity and generalization.

### Predict before fitting

1. Which model should have the largest training–validation gap: a single deep tree, Random Forest, or Gradient Boosting with shallow trees?
2. Does a tree-based model require feature standardisation for its split rules?
3. Why must the test set remain unused in the next comparison?

In [ ]:
comparison_models = {
    'Decision tree': DecisionTreeClassifier(
        max_depth=None,
        random_state=RANDOM_STATE,
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=1,
        random_state=RANDOM_STATE,
    ),
    'Histogram Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=RANDOM_STATE,
    ),
}


def evaluate_classifier(model, X_train, y_train, X_eval, y_eval):
    model.fit(X_train, y_train)

    train_probability = model.predict_proba(X_train)[:, 1]
    eval_probability = model.predict_proba(X_eval)[:, 1]
    train_prediction = (train_probability >= 0.5).astype(int)
    eval_prediction = (eval_probability >= 0.5).astype(int)

    return {
        'train accuracy': accuracy_score(y_train, train_prediction),
        'validation accuracy': accuracy_score(y_eval, eval_prediction),
        'validation F1': f1_score(y_eval, eval_prediction),
        'validation ROC-AUC': roc_auc_score(y_eval, eval_probability),
        'validation log loss': log_loss(y_eval, eval_probability),
    }

validation_rows = []

for name, model in comparison_models.items():
    metrics = evaluate_classifier(
        model,
        X_train,
        y_train,
        X_valid,
        y_valid,
    )
    validation_rows.append({'model': name, **metrics})

validation_results = pd.DataFrame(validation_rows).set_index('model')
validation_results.round(3)

### Activity 4 - Compare by more than accuracy

For each model, comment on:

- training versus validation accuracy,
- validation ROC-AUC,
- validation log loss,
- model complexity or training strategy.

Then write a preliminary recommendation. Do **not** use the test set yet.

**Preliminary recommendation:**

## 6. Select the number of stages using validation loss

`GradientBoostingClassifier` exposes predictions after every stage. This allows us to ask:

> At which stage is validation log loss lowest?

The training curve will usually continue decreasing. The validation curve may eventually flatten or rise.

In [ ]:
gb_full = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=1,
    random_state=RANDOM_STATE,
)
gb_full.fit(X_train, y_train)

train_log_loss_curve = [
    log_loss(y_train, probability)
    for probability in gb_full.staged_predict_proba(X_train)
]
valid_log_loss_curve = [
    log_loss(y_valid, probability)
    for probability in gb_full.staged_predict_proba(X_valid)
]

BEST_STAGE = int(np.argmin(valid_log_loss_curve) + 1)

print('Best validation stage:', BEST_STAGE)
print('Best validation log loss:', round(min(valid_log_loss_curve), 4))

plt.figure(figsize=(10, 5.5))
plt.plot(
    range(1, len(train_log_loss_curve) + 1),
    train_log_loss_curve,
    label='Training log loss',
)
plt.plot(
    range(1, len(valid_log_loss_curve) + 1),
    valid_log_loss_curve,
    label='Validation log loss',
)
plt.axvline(BEST_STAGE, linestyle='--', label=f'Best validation stage: {BEST_STAGE}')
plt.xlabel('Boosting stage')
plt.ylabel('Log loss')
plt.title('Training and validation loss across stages')
plt.legend()
plt.show()

### Activity 5 — Diagnose the stage curve

Complete:

> Training loss continues to __________ because __________.  
> Validation loss reaches its minimum near stage __________.  
> Adding many more stages after this point risks __________.  
> The stage was selected using the __________ set, not the test set.

Why is the minimum validation loss a more defensible stopping point than the minimum training loss?

## 7. Final retraining and one test-set audit

After choosing the stage count, combine training and validation data, fit the selected model, and evaluate the untouched test set once.

In [ ]:
selected_gb = GradientBoostingClassifier(
    n_estimators=BEST_STAGE,
    learning_rate=0.05,
    max_depth=1,
    random_state=RANDOM_STATE,
)
selected_gb.fit(X_train_full, y_train_full)

test_probability = selected_gb.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

final_test_result = pd.Series({
    'test accuracy': accuracy_score(y_test, test_prediction),
    'test F1': f1_score(y_test, test_prediction),
    'test ROC-AUC': roc_auc_score(y_test, test_probability),
    'test log loss': log_loss(y_test, test_probability),
})

final_test_result.round(3)

### Verification checkpoint

1. Is the test result reasonably consistent with the validation result?
2. What would it mean if the test result were dramatically worse?
3. Why should you not repeatedly change the model after seeing this test result?

## 8. Optimised boosting implementation and early stopping

`HistGradientBoostingClassifier` groups continuous feature values into bins before searching for splits. This can make training more efficient on larger tabular datasets.

It can also stop automatically when its internal validation score no longer improves.

In [ ]:
hist_model = comparison_models['Histogram Gradient Boosting']

print('Iterations actually used:', hist_model.n_iter_)
print('Maximum allowed iterations:', hist_model.max_iter)
print('Stopped early:', hist_model.n_iter_ < hist_model.max_iter)

### Stop and explain

1. Why can binning make split search faster?
2. What is the benefit of early stopping?
3. Why does automatic early stopping still depend on choices such as validation fraction and patience?


## 9. Reading: Interpreting models with Permutation Importance

### Why do we need feature importance?

Complex machine learning models, such as Gradient Boosting, can achieve high predictive performance, but their decisions are often difficult to interpret.

Unlike a small decision tree, an ensemble of hundreds of trees cannot easily be understood by inspecting individual splits.

**Permutation Importance** provides a simple, model-independent method for estimating how much a trained model relies on individual input features.

### How does Permutation Importance work?

The main idea is straightforward: **if a feature is important for prediction, randomly shuffling its values should make the model perform worse.**

The procedure consists of four steps:

1. Evaluate the trained model on the original validation dataset and record its performance.
2. Select one feature and randomly shuffle its values across observations, keeping all other features unchanged.
3. Evaluate the same model on the modified dataset.
4. Measure the decrease in predictive performance.

The model is **not retrained** during this process.

For a performance metric where higher values are better (such as accuracy), importance is calculated as:

$$
I_j = S_{\mathrm{original}} - S_{\mathrm{permuted},j}
$$

where:

- $I_j$ is the permutation importance of feature $j$,
- $S_{\mathrm{original}}$ is the original validation score,
- $S_{\mathrm{permuted},j}$ is the score after permuting feature $j$.

For a loss function where lower values are better, such as log loss, we instead measure the increase in loss.

### Example

Suppose we have trained a classifier to predict whether a house is expensive.

The original validation accuracy is **90%**.

After independently permuting each feature, we obtain:

| Permuted feature | Accuracy after permutation | Importance |
|---|---|---|
| Living area | 65% | 0.25 |
| Building age | 82% | 0.08 |
| Number of rooms | 88% | 0.02 |
| Random identifier | 90% | 0.00 |

We can interpret these results as follows:

- **Living area** is highly important: disrupting its values causes a substantial performance decrease.
- **Building age** contributes to the model's predictions, but less strongly.
- **Number of rooms** has a relatively small measured contribution.
- **Random identifier** has no measurable importance in this experiment.

In practice, permutation is repeated several times because different random shuffles may produce different results.

The mean importance and its standard deviation help us assess the stability of the estimate.

### Important limitations

Permutation Importance measures how much a **specific trained model** depends on a feature for its predictions.

However, there are several limitations:

- **Correlation between features:** If two features contain similar information, permuting one may have little effect because the model can still use the other.
- **Unrealistic combinations:** Randomly shuffling a feature can create combinations of values that rarely occur in real data.
- **Dependence on the evaluation metric:** A feature may appear important for accuracy but less important for log loss.
- **No causal interpretation:** High permutation importance does not prove that changing the feature would cause a change in the target variable.

Importance can also be close to zero or negative. Negative importance means that the model performed slightly better after the feature was shuffled, which may reflect noise, sampling variability, or harmful reliance on that feature.

### Why use validation data?

Permutation Importance should be calculated on data that was not used to train the model.

Using training data may produce misleading conclusions because the model could have learned patterns that do not generalize.

A separate validation dataset allows us to estimate which features are useful for predictions on unseen observations.

### Check your understanding

1. Why should we shuffle only one feature at a time?
2. What does it mean if validation accuracy drops from 90% to 60% after permuting a feature?
3. Why might two highly correlated features both receive low permutation importance?
4. Does high permutation importance mean that a feature causes the predicted outcome?


In [ ]:
importance = permutation_importance(
    selected_gb,
    X_test,
    y_test,
    scoring='roc_auc',
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_table = pd.DataFrame({
    'feature': X.columns,
    'mean ROC-AUC decrease': importance.importances_mean,
    'standard deviation': importance.importances_std,
}).sort_values('mean ROC-AUC decrease', ascending=False)

display(importance_table.head(10).round(4))

top_importance = importance_table.head(10).sort_values('mean ROC-AUC decrease')
plt.figure(figsize=(9, 5.5))
plt.barh(
    top_importance['feature'],
    top_importance['mean ROC-AUC decrease'],
)
plt.xlabel('Mean ROC-AUC decrease')
plt.ylabel('Feature')
plt.title('Top permutation importances for the selected boosting model')
plt.show()

# End-of-lab self-check

Before finishing, check that you can:

- [ ] explain what residuals represent in boosting,
- [ ] describe one boosting update,
- [ ] explain why shallow trees can form a complex additive model,
- [ ] describe the interaction between learning rate and number of trees,
- [ ] read training and validation stage curves,
- [ ] identify a defensible early-stopping point,
- [ ] compare Random Forest and Gradient Boosting conceptually,
- [ ] explain the purpose of Histogram Gradient Boosting,
- [ ] keep validation and test roles separate,
- [ ] interpret permutation importance without causal language,
- [ ] use AI questions to improve an evidence-based diagnosis.

Exercise 8 will focus on dimensionality reduction and latent-space visualisation.